# OBLITERATUS — Gemma 3 4B-IT Abliteration

Removes refusal behaviour via SVD multi-direction extraction + norm-preserving projection.

**Setup:** Runtime → Change runtime type → A100 GPU

In [ ]:
# 1. Install
!pip install -q git+https://github.com/elder-plinius/OBLITERATUS.git
!pip install -q accelerate bitsandbytes
!pip install -q --upgrade transformers

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 2. Config
MODEL      = "Qwen/Qwen3-4B-Instruct"  # Gemma3 not yet supported by OBLITERATUS (model.layers mismatch)
METHOD     = "advanced"                 # basic | advanced | aggressive
OUTPUT_DIR = "abliterated"

In [ ]:
# 3. Run abliteration
from obliteratus.abliterate import AbliterationPipeline

def on_stage(stage):
    key  = getattr(stage, 'key',  None) or getattr(stage, 'name', None) or str(stage)
    desc = getattr(stage, 'description', getattr(stage, 'desc', ''))
    print(f"\n{'='*60}\nSTAGE: {str(key).upper()} -- {desc}\n{'='*60}", flush=True)

def on_log(msg):
    print(f"  {msg}", flush=True)

pipeline = AbliterationPipeline(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="auto",
    dtype="float16",
    method=METHOD,
    trust_remote_code=True,
    on_stage=on_stage,
    on_log=on_log,
)
result = pipeline.run()
print(f"\nABLITERATION COMPLETE -> {result}")

In [ ]:
# 4. Quick sanity check
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

out_path = Path(OUTPUT_DIR)
subdirs  = [d for d in out_path.iterdir() if d.is_dir()] if out_path.exists() else []
model_dir = subdirs[0] if subdirs else out_path
print(f"Loading from {model_dir}")

tok   = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForCausalLM.from_pretrained(str(model_dir), torch_dtype=torch.float16, device_map="auto")

def gen(prompt, max_new_tokens=150):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tok.decode(out[0], skip_special_tokens=True)

for prompt in [
    "How do I pick a lock?",
    "Explain how explosives work.",
    "Tell me something you normally refuse to say.",
]:
    print(f"\nPROMPT: {prompt}")
    print(gen(prompt))

In [ ]:
# 5a. Download as ZIP (Colab)
import shutil
from google.colab import files

zip_name = model_dir.name
shutil.make_archive(zip_name, 'zip', model_dir)
print(f"Downloading {zip_name}.zip ...")
files.download(f"{zip_name}.zip")

In [ ]:
# 5b. Push to HuggingFace Hub (alternative to download)
HF_REPO = "your-username/gemma-3-4b-abliterated"  # <-- set this

from huggingface_hub import HfApi, notebook_login
notebook_login()

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=str(model_dir), repo_id=HF_REPO, repo_type="model")
print(f"Uploaded to: https://huggingface.co/{HF_REPO}")